In [1]:
!pip -q install transformers datasets

Import libraries

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

import torch

print("Libraries imported successfully!")

Libraries imported successfully!


Create Sample Dataset

In [3]:
data = {
    "text": [
        "I really enjoyed this movie.",
        "The movie was excellent and amazing.",
        "I loved the product.",
        "This is a wonderful experience.",
        "The service was very good.",
        "I am happy with this purchase.",
        "The movie was boring.",
        "I did not like this movie.",
        "The product was terrible.",
        "This was a bad experience.",
        "The service was very poor.",
        "I am disappointed with this purchase.",
        "The quality is excellent.",
        "I absolutely love this product.",
        "The experience was fantastic.",
        "This is the worst product.",
        "I hate this movie.",
        "The service was disappointing.",
        "The product works perfectly.",
        "I am unhappy with the result."
    ],

    "label": [
        1, 1, 1, 1, 1, 1,
        0, 0, 0, 0, 0, 0,
        1, 1, 1,
        0, 0, 0,
        1, 0
    ]
}

df = pd.DataFrame(data)

print(df.head())
print("\nDataset shape:", df.shape)

                                   text  label
0          I really enjoyed this movie.      1
1  The movie was excellent and amazing.      1
2                  I loved the product.      1
3       This is a wonderful experience.      1
4            The service was very good.      1

Dataset shape: (20, 2)


Split Dataset

In [4]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Training samples:", len(train_texts))
print("Testing samples:", len(test_texts))

Training samples: 16
Testing samples: 4


Load BERT Tokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("BERT tokenizer loaded successfully!")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BERT tokenizer loaded successfully!


Tokenize the Dataset

In [6]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

print("Tokenization completed!")
print("Number of training samples:", len(train_encodings["input_ids"]))

Tokenization completed!
Number of training samples: 16


Create Dataset Class

In [7]:
class SentimentDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [8]:
train_dataset = SentimentDataset(train_encodings, train_labels)
test_dataset = SentimentDataset(test_encodings, test_labels)

print("BERT datasets created successfully!")

BERT datasets created successfully!


Load Pre-trained BERT Model

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

print("Pre-trained BERT model loaded successfully!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Pre-trained BERT model loaded successfully!


Define Training Arguments

In [11]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    logging_steps=1,
    report_to="none"
)

print("Training arguments configured!")

Training arguments configured!


Create Trainer

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

print("Trainer created successfully!")

Trainer created successfully!


Train BERT Model

In [13]:
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.737844
2,0.767735
3,0.597088
4,0.596710
5,0.612184
6,0.622028
7,0.600487
8,0.499235
9,0.529015
10,0.539833


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12, training_loss=0.591454287370046, metrics={'train_runtime': 41.8491, 'train_samples_per_second': 1.147, 'train_steps_per_second': 0.287, 'total_flos': 221999952960.0, 'train_loss': 0.591454287370046, 'epoch': 3.0})

Evaluate the Model

In [14]:
results = trainer.evaluate()

print("Evaluation Results:")
print(results)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step
0.548435,0.672129,12


Evaluation Results:
{'eval_loss': 0.6721290349960327}


Make predictions

In [15]:
predictions = trainer.predict(test_dataset)

predicted_labels = np.argmax(
    predictions.predictions,
    axis=1
)

print("Predicted labels:")
print(predicted_labels)

print("\nActual labels:")
print(np.array(test_labels))

Predicted labels:
[0 0 0 0]

Actual labels:
[0 0 1 1]


Calculate Accuracy

In [16]:
accuracy = accuracy_score(test_labels, predicted_labels)

print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)

Accuracy: 0.5
Accuracy (%): 50.0


Classification Report

In [17]:
print(
    classification_report(
        test_labels,
        predicted_labels,
        target_names=["Negative", "Positive"],
        zero_division=0
    )
)

              precision    recall  f1-score   support

    Negative       0.50      1.00      0.67         2
    Positive       0.00      0.00      0.00         2

    accuracy                           0.50         4
   macro avg       0.25      0.50      0.33         4
weighted avg       0.25      0.50      0.33         4



Confusion matrix

In [18]:
cm = confusion_matrix(test_labels, predicted_labels)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[2 0]
 [2 0]]


Test BERT with Your Own Sentence

In [20]:
text = ["This product is terrible and disappointing."]

inputs = tokenizer(
    text,
    return_tensors="pt",
    padding=True,
    truncation=True
)

with torch.no_grad():
    outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

if prediction == 1:
    print("Sentiment: Positive")
else:
    print("Sentiment: Negative")

Sentiment: Negative
